# 101_paper_tables_1_4

Build Table 1/2/3/4 in the **new paper format**. This notebook keeps existing analysis notebooks unchanged and adds the full table set.

In [ ]:
import os, sys, pathlib
import torch
import pandas as pd


def _find_project_root():
    here = pathlib.Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src").is_dir():
            return p
    cand = pathlib.Path("/content/econml")
    if (cand / "src").is_dir():
        return cand
    raise RuntimeError("Could not find project root containing src/.")

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelParams
from src.paper_tables import build_table1_calibration, build_paper_tables_2_4, _paper_moments_wide_table
from src.table2_builder import build_table0

ARTIFACTS_ROOT = str(PROJECT_ROOT / "artifacts")
COMPUTE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Compute device:", COMPUTE_DEVICE)
print("Artifacts root:", ARTIFACTS_ROOT)


In [ ]:
# ---- Table 1 (calibration) ----
params = ModelParams(device=COMPUTE_DEVICE, dtype=torch.float64).to_torch()
df_t1 = build_table1_calibration(params)
display(df_t1)

p1 = os.path.join(ARTIFACTS_ROOT, "table1_calibration.csv")
df_t1.to_csv(p1, index=False)
print("Saved:", p1)


In [ ]:
# ---- Table 2/3/4 from runs ----
# Paper/author mode: use simulation-conditional SSS (long-run conditional means by regime).
tables = build_paper_tables_2_4(
    ARTIFACTS_ROOT,
    device=COMPUTE_DEVICE,
    dtype=torch.float64,
    use_selected=True,
    sss_source="sim_conditional",
)

for key in ["table2", "table3", "table4"]:
    print("\n===", key.upper(), "===")
    df = tables[key]
    display(df)
    out = os.path.join(ARTIFACTS_ROOT, f"{key}_sim_conditional.csv")
    df.to_csv(out, index=False)
    print("Saved:", out)


In [ ]:
# Optional diagnostic: fixed-point variant for comparison
# NOTE: build_paper_tables_2_4 always enforces sim_conditional, so for true fixed-point
# diagnostics we build table0 directly and reshape to paper layout.
try:
    df_diag = build_table0(
        ARTIFACTS_ROOT,
        device=COMPUTE_DEVICE,
        dtype=torch.float64,
        use_selected=True,
        include_rules=True,
        include_zlb=True,
        sss_source="fixed_point",
    )
except FileNotFoundError as e:
    if "zlb" not in str(e).lower():
        raise
    print("[diagnostic fixed_point] WARNING: missing ZLB runs; Table 4 will contain NaN placeholders.")
    df_diag = build_table0(
        ARTIFACTS_ROOT,
        device=COMPUTE_DEVICE,
        dtype=torch.float64,
        use_selected=True,
        include_rules=True,
        include_zlb=False,
        sss_source="fixed_point",
    )

tables_diag = {
    "table2": _paper_moments_wide_table(
        df_diag,
        policies=[
            ("flex", "Flex. prices"),
            ("taylor", "Taylor rule"),
            ("mod_taylor", "Mod. Taylor rule"),
        ],
    ),
    "table3": _paper_moments_wide_table(
        df_diag,
        policies=[
            ("discretion", "Discretion"),
            ("commitment", "Commitment"),
        ],
    ),
    "table4": _paper_moments_wide_table(
        df_diag,
        policies=[
            ("taylor_zlb", "Taylor rule"),
            ("mod_taylor_zlb", "Mod. Taylor rule"),
            ("discretion_zlb", "Discretion"),
            ("commitment_zlb", "Commitment"),
        ],
    ),
}

for key in ["table2", "table3", "table4"]:
    out = os.path.join(ARTIFACTS_ROOT, f"{key}_fixed_point.csv")
    tables_diag[key].to_csv(out, index=False)
    print("Saved:", out)
